# 01 — Dataset: coleta, organização e legendas (BLIP)

**Ateliê Generativo — Arquitetura Modernista de Brasília**

Este notebook faz a Etapa 1 do projeto:
1. Traz as imagens do repositório GitHub para o Colab
2. Gera legendas *rascunho* automaticamente com o BLIP
3. Exporta essas legendas para revisão manual (obrigatória)
4. Depois de revisadas, valida e monta o `legendas.txt` final

⚠️ Antes de rodar: ative a GPU em **Ambiente de execução → Alterar tipo de ambiente → T4 GPU**.

## Passo 1 — Clonar o repositório

Traz as imagens e o `fontes.csv` que já estão versionados no GitHub.

In [ ]:
!git clone https://github.com/lramosc1512/atelie-generativo-LeonidIA.git
%cd atelie-generativo-LeonidIA
!ls dados/imagens | head
!wc -l dados/fontes.csv

## Passo 2 — Instalar dependências

In [ ]:
!pip -q install transformers pillow pandas

## Passo 3 — Carregar o modelo BLIP

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {device}")

proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

## Passo 4 — Gerar legendas rascunho para todas as imagens

O BLIP gera uma descrição genérica em inglês. **Isso não é a legenda final** — é só o ponto de partida para você revisar no Passo 6.
Nada aqui ainda leva o token `estilo_brasilia,` — isso só entra depois da revisão.

In [ ]:
IMG_DIR = Path("dados/imagens")
extensoes = {".jpg", ".jpeg", ".png"}
arquivos = sorted([p for p in IMG_DIR.iterdir() if p.suffix.lower() in extensoes])
print(f"{len(arquivos)} imagens encontradas.")

rascunhos = []
for caminho in arquivos:
    img = Image.open(caminho).convert("RGB")
    entradas = proc(img, return_tensors="pt").to(device)
    saida = blip.generate(**entradas, max_new_tokens=40)
    legenda_bruta = proc.decode(saida[0], skip_special_tokens=True)
    rascunhos.append({"arquivo": caminho.name, "legenda_blip_en": legenda_bruta, "legenda_revisada": ""})
    print(f"{caminho.name}: {legenda_bruta}")

df_rascunho = pd.DataFrame(rascunhos)
df_rascunho.to_csv("dados/legendas_rascunho.csv", index=False)
df_rascunho.head()

## Passo 5 — Baixar o rascunho para revisão manual

A revisão humana é **obrigatória** e será verificada pelo professor. O BLIP erra bastante com arquitetura (confunde prédios com "pontes", "estádios", etc.), então não pule esta etapa.

**Como revisar:**
1. Baixe o `legendas_rascunho.csv` gerado abaixo
2. Abra no Google Sheets ou Excel
3. Para cada linha, preencha a coluna `legenda_revisada` em português, descrevendo a imagem de verdade (ignore a coluna em inglês do BLIP, use só como referência)
4. **Comece cada legenda revisada com o token do estilo:** `estilo_brasilia, ...`
   - Exemplo: `estilo_brasilia, fachada curva do Palácio do Itamaraty refletida no espelho d'água`
5. Salve como CSV e reenvie no Passo 6

In [ ]:
from google.colab import files
files.download("dados/legendas_rascunho.csv")

## Passo 6 — Reenviar o CSV revisado

Após editar no Sheets/Excel, exporte como CSV e suba o arquivo aqui.

In [ ]:
from google.colab import files
enviado = files.upload()  # selecione o CSV revisado
nome_arquivo_revisado = list(enviado.keys())[0]
df_revisado = pd.read_csv(nome_arquivo_revisado)
df_revisado.head()

## Passo 7 — Validação

Antes de gerar o `legendas.txt` final, checamos três coisas automaticamente:
1. Toda imagem tem uma legenda revisada preenchida (não vazia)
2. Toda legenda começa com o token `estilo_brasilia,`
3. Todo arquivo listado em `legendas_revisado.csv` também existe em `dados/fontes.csv`

Se algo falhar, o notebook avisa exatamente qual linha corrigir — **não prossiga até os três testes passarem.**

In [ ]:
TOKEN = "estilo_brasilia,"
problemas = []

# 1) legendas vazias
vazias = df_revisado[df_revisado["legenda_revisada"].isna() | (df_revisado["legenda_revisada"].str.strip() == "")]
for _, linha in vazias.iterrows():
    problemas.append(f'Legenda vazia: {linha["arquivo"]}')

# 2) token no inicio
com_legenda = df_revisado.dropna(subset=["legenda_revisada"])
sem_token = com_legenda[~com_legenda["legenda_revisada"].str.strip().str.startswith(TOKEN)]
for _, linha in sem_token.iterrows():
    problemas.append(f'Falta o token "{TOKEN}" no inicio: {linha["arquivo"]}')

# 3) arquivo existe em fontes.csv
fontes = pd.read_csv("dados/fontes.csv")
nomes_fontes = set(fontes["nome_arquivo"])
nomes_legendas = set(df_revisado["arquivo"])
faltando_em_fontes = nomes_legendas - nomes_fontes
for nome in faltando_em_fontes:
    problemas.append(f'Arquivo sem entrada correspondente em fontes.csv: {nome}')

if problemas:
    print(f"{len(problemas)} problema(s) encontrado(s):\n")
    for p in problemas:
        print(f"  - {p}")
else:
    print("Tudo certo! Pode seguir para o Passo 8.")

## Passo 8 — Gerar o `legendas.txt` final

In [ ]:
assert not problemas, "Corrija os problemas do Passo 7 antes de continuar."

with open("dados/legendas.txt", "w", encoding="utf-8") as f:
    for _, linha in df_revisado.iterrows():
        f.write(f'{linha["arquivo"]},{linha["legenda_revisada"].strip()}\n')

print("dados/legendas.txt gerado com sucesso.")
!head dados/legendas.txt

## Passo 9 — Commitar no repositório

O Colab não tem seu token do GitHub configurado por padrão — o mais simples é **baixar o `legendas.txt` e commitar do seu computador**, pelo Git Bash, como você já vem fazendo:

```bash
cd caminho/para/atelie-generativo-LeonidIA
git add dados/legendas.txt
git commit -m "Adiciona legendas revisadas manualmente via BLIP"
git push
```

In [ ]:
from google.colab import files
files.download("dados/legendas.txt")